Phase 1: Data Sourcing & Base Training


In [20]:
!pip install ultralytics opencv-python onnxruntime

In [21]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
%cd /content/drive/MyDrive/Colab Notebooks/PPE
model.train(data="data.yaml",epochs=50,imgsz=640,batch=16,device=0)

/content/drive/MyDrive/Colab Notebooks/PPE
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7877fc2a3740>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [26]:
import os
model = YOLO("runs/detect/train/weights/best.pt")

metrics = model.val()

print(metrics.box.map)
print(metrics.box.map50)


Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 29.8±37.1 MB/s, size: 78.6 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/PPE/dataset/valid/labels.cache... 62 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 83/83 19.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2it/s 4.9s
                   all         83        230      0.258      0.239      0.232      0.123
                Jacket         18         34      0.616      0.824      0.708      0.477
              hard_hat          3          5          0          0          0          0
                Gloves         15         31      0.211      0.161      0.201      0.102
                 Shoes         37         80      0.217     0.0125      0.046     0.0236
     Hard_hat_required

In [27]:
pt_size = os.path.getsize("runs/detect/train/weights/best.pt") / (1024 * 1024)
print("PT size:", round(pt_size, 2), "MB")


PT size: 5.96 MB


Phase 2: Edge Conversion & Quantization

In [25]:
model.export(format="onnx",half=True)



Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 13, 8400) (6.0 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: converting to FP16...
ONNX: export success ✅ 1.9s, saved as 'runs/detect/train/weights/best.onnx' (5.9 MB)

Export complete (2.3s)
Results saved to /content/drive/MyDrive/Colab Notebooks/PPE/runs/detect/train/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/train/weights/best.onnx imgsz=640 half
Validate:        yolo val task=detect model=runs/detect/train/weights/best.onnx imgsz=640 data=data.yaml half 
Visualize:       https://netron.app


'runs/detect/train/weights/best.onnx'

In [30]:
model = YOLO(
    "/content/drive/MyDrive/Colab Notebooks/PPE/runs/detect/train/weights/best.onnx"
)

metrics = model.val(
    data="/content/drive/MyDrive/Colab Notebooks/PPE/data.yaml"
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)

onnx_size = os.path.getsize(
    "/content/drive/MyDrive/Colab Notebooks/PPE/runs/detect/train/weights/best.onnx"
) / (1024 * 1024)

print("ONNX size:", round(onnx_size, 2), "MB")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/Colab Notebooks/PPE/runs/detect/train/weights/best.onnx for ONNX Runtime inference...
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.26.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 27.8±24.3 MB/s, size: 68.8 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/PPE/dataset/valid/labels.cache... 62 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 83/83 14.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 8.2it/s 10.2s
                   all         83        230      0.193      0.263      0.218      0.116
                Jacket         18         34      0.546      0.824      0.709      0.496
              hard_hat          3          5          0          0   

Phase 3: The Inference Script & Metrics

In [ ]:
import time
start = time.time()

for _ in range(100):
    model("/content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg")

fps = 100 / (time.time() - start)

print(fps)


image 1/1 /content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg: 640x448 1 Gloves, 41.2ms
Speed: 2.2ms preprocess, 41.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg: 640x448 1 Gloves, 10.8ms
Speed: 2.7ms preprocess, 10.8ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg: 640x448 1 Gloves, 6.1ms
Speed: 2.2ms preprocess, 6.1ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg: 640x448 1 Gloves, 6.0ms
Speed: 1.5ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 448)

image 1/1 /content/drive/MyDrive/Colab Notebooks/PPE/dataset/test/images/p34.jpeg: 640x448 1 Gloves, 6.0ms
Speed: 1.5ms preprocess, 6.0ms inference, 1.1ms postprocess per image at